# Fragrantica Parfüm Puanı Tahmin Projesi
**MBP 227 – Veri Bilimi ve Makine Öğrenmesi | Final Projesi**

**Veri Kaynağı:** `fra_cleaned.csv`

**Hedef:** Üst nota, alt nota ve marka bilgisine göre kullanıcı puanını tahmin etmek

**Model Türü:** Regresyon



## 1. Kütüphaneler ve Veri Yükleme

In [8]:
%matplotlib tk
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


# Veriyi yükle
df = pd.read_csv('fra_cleaned.csv', sep=';', encoding='latin1')

# Rating Value sütununu ondalık virgülden noktaya çevir
df['Rating Value'] = df['Rating Value'].astype(str).str.replace(',', '.').astype(float)

print("Veri yüklendi:", df.shape)
df.head(3)

## 2. Keşifsel Veri Analizi (EDA)

In [36]:
# Temel bilgiler
print("Boyut:", df.shape)
print("\nVeri tipleri:")
print(df.dtypes)
print("\nİlk 5 satır:")
df.head()

In [37]:
# Temel istatistikler
df.describe()

In [38]:
# Eksik değer analizi
print("Eksik değer sayıları:")
print(df.isnull().sum())
print("\nBenzersiz değer sayıları:")
print(df.nunique())

In [39]:
# Sayısal sütunlar arası korelasyon
num_cols = df.select_dtypes(include='number')
print("Korelasyon Matrisi:")
num_cols.corr()

## 3. Eksik Değer Analizi ve Temizleme

In [40]:
from pandas.api.types import is_numeric_dtype

print("Öncesi – Satır:", df.shape[0], "| Sütun:", df.shape[1])

# Eksiklik oranı yüksek sütunları bul (%40 üzeri)
threshold = 0.40
yuksek_eksik = [c for c in df.columns if df[c].isna().mean() > threshold]
print("Yüksek eksiklikli sütunlar (>%40):", yuksek_eksik)
df.drop(columns=yuksek_eksik, inplace=True)

# Sayısal sütunları medyan ile, diğerlerini mod ile doldur
for col in df.columns:
    if df[col].isna().sum() == 0:
        continue

    if is_numeric_dtype(df[col]):
        # Eğer kolon sayısal ise medyan kullan
        df[col].fillna(df[col].median(), inplace=True)
    else:
        # Sayısal değilse mod kullan
        df[col].fillna(df[col].mode()[0], inplace=True)

print("Sonrası – Satır:", df.shape[0], "| Sütun:", df.shape[1])
print("Kalan eksik:", df.isnull().sum().sum())

## 4. Aykırı Değer Analizi ve Temizleme



In [41]:
# IQR yöntemiyle Rating Value aykırı değerleri
Q1 = df['Rating Value'].quantile(0.25)
Q3 = df['Rating Value'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

print(f"IQR: {IQR:.3f} | Alt sınır: {alt_sinir:.3f} | Üst sınır: {ust_sinir:.3f}")
aykiri = df[(df['Rating Value'] < alt_sinir) | (df['Rating Value'] > ust_sinir)]
print(f"Aykırı değer sayısı (IQR): {len(aykiri)}")

# Z-Score yöntemi
from scipy import stats
z = np.abs(stats.zscore(df['Rating Value']))
print(f"Aykırı değer sayısı (Z-Score >3): {(z > 3).sum()}")

# IQR sınırları içine filtrele
df = df[(df['Rating Value'] >= alt_sinir) & (df['Rating Value'] <= ust_sinir)]
print("Temizleme sonrası satır sayısı:", len(df))

## 5. Seaborn Görselleştirme

In [20]:
# 1 - Histplot: Rating Value dağılımı
plt.figure(figsize=(8,4))
sns.histplot(df['Rating Value'], kde=True, bins=30, color='steelblue')
plt.title('Kullanıcı Puanlarının Dağılımı')
plt.xlabel('Puan (1-5)')
plt.ylabel('Frekans')
plt.tight_layout()
plt.show()


In [10]:
# 2 - Boxplot: Cinsiyete göre puan
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x='Gender', y='Rating Value', palette='pastel')
plt.title('Cinsiyete Göre Puan Dağılımı')
plt.xlabel('Cinsiyet')
plt.ylabel('Puan')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [11]:
# 3 - Heatmap: Korelasyon matrisi
plt.figure(figsize=(7,5))
num_df = df.select_dtypes(include='number')
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Sayısal Değişkenler Korelasyon Matrisi')
plt.tight_layout()
plt.show()


In [21]:
# 4 - Pairplot: Sayısal değişkenler
sns.pairplot(df[['Rating Value', 'Rating Count', 'Year']].dropna(),
             plot_kws={'alpha': 0.3})
plt.suptitle('Sayısal Değişkenler Pairplot', y=1.02)
plt.show()


In [24]:
# 5 - Barplot: İlk 10 markanın ortalama puanı
top_brands = df.groupby('Brand')['Rating Value'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,4))
sns.barplot(x=top_brands.index, y=top_brands.values, palette='viridis')
plt.title('En Yüksek Ortalama Puanlı 10 Marka')
plt.xlabel('Marka')
plt.ylabel('Ortalama Puan')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [15]:
# 6 - Scatterplot: Rating Count vs Rating Value
plt.figure(figsize=(8,5))
sample = df.sample(2000, random_state=42)
sns.scatterplot(data=sample, x='Rating Count', y='Rating Value',
                hue='Gender', size='Rating Count',
                sizes=(20, 200), alpha=0.6)
plt.title('Oy Sayısı vs Puan (2000 örnek)')
plt.xlabel('Oy Sayısı')
plt.ylabel('Puan')
plt.tight_layout()
plt.show()


In [16]:
# 7 - Violinplot: Ülkeye göre puan (ilk 5 ülke)
top5_countries = df['Country'].value_counts().head(5).index
plt.figure(figsize=(10,5))
sns.violinplot(data=df[df['Country'].isin(top5_countries)],
               x='Country', y='Rating Value', palette='muted')
plt.title('İlk 5 Ülkenin Puan Dağılımı')
plt.xlabel('Ülke')
plt.ylabel('Puan')
plt.tight_layout()
plt.show()


In [22]:
# 8 - Countplot: Cinsiyete göre parfüm sayısı
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='Gender', order=df['Gender'].value_counts().index, palette='Set2')
plt.title('Cinsiyete Göre Parfüm Sayısı')
plt.xlabel('Cinsiyet')
plt.ylabel('Adet')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [23]:
# 9 - Lineplot: Yıla göre ortalama puan trendi
year_avg = df.groupby('Year')['Rating Value'].mean().reset_index()
year_avg = year_avg[(year_avg['Year'] >= 2000) & (year_avg['Year'] <= 2024)]
plt.figure(figsize=(10,4))
sns.lineplot(data=year_avg, x='Year', y='Rating Value', marker='o', color='coral')
plt.title('Yıllara Göre Ortalama Kullanıcı Puanı')
plt.xlabel('Yıl')
plt.ylabel('Ortalama Puan')
plt.tight_layout()
plt.show()


In [19]:
# 10 - KDE Plot: Puan yoğunluğu cinsiyete göre
plt.figure(figsize=(8,4))
for gender in df['Gender'].value_counts().head(3).index:
    subset = df[df['Gender'] == gender]['Rating Value']
    sns.kdeplot(subset, label=gender, fill=True, alpha=0.4)
plt.title('Cinsiyete Göre Puan Yoğunluğu (KDE)')
plt.xlabel('Puan')
plt.ylabel('Yoğunluk')
plt.legend()
plt.tight_layout()
plt.show()


## 6. GroupBy & Koşullu Sorgular

In [52]:
# Sorgu 1 – groupby + mean(): Ülke bazında ortalama puan
df.groupby('Country')['Rating Value'].mean().sort_values(ascending=False).head(10)

In [53]:
# Sorgu 2 – groupby + count(): Her markada kaç parfüm var?
df.groupby('Brand')['Perfume'].count().sort_values(ascending=False).head(10)

In [54]:
# Sorgu 3 – groupby + agg(): Marka bazında min, max, ortalama puan
df.groupby('Brand')['Rating Value'].agg(['min', 'max', 'mean']).sort_values('mean', ascending=False).head(10)

In [55]:
# Sorgu 4 – groupby + sum(): Cinsiyete göre toplam oy sayısı
df.groupby('Gender')['Rating Count'].sum().sort_values(ascending=False)

In [56]:
# Sorgu 5 – groupby + std(): Markalar arası puan standart sapması
df.groupby('Brand')['Rating Value'].std().sort_values(ascending=False).head(10)

In [57]:
# Sorgu 6 – loc[koşul]: Puanı 4.5 üzerindeki parfümler
df.loc[df['Rating Value'] > 4.5, ['Perfume', 'Brand', 'Rating Value']].head(10)

In [58]:
# Sorgu 7 – query(): Fransız ve puanı 4.0'dan yüksek parfümler
df.query("Country == 'France' and `Rating Value` > 4.0")[['Perfume', 'Brand', 'Rating Value']].head(10)

In [59]:
# Sorgu 8 – groupby + filter(): En az 50 parfümü olan markalar
df.groupby('Brand').filter(lambda x: len(x) >= 50)['Brand'].value_counts().head(10)

In [60]:
# Sorgu 9 – groupby + transform(): Her parfüme kendi markasının ortalama puanını ekle
df['Marka_Ort_Puan'] = df.groupby('Brand')['Rating Value'].transform('mean')
df[['Perfume', 'Brand', 'Rating Value', 'Marka_Ort_Puan']].head(5)

In [61]:
# Sorgu 10 – sort_values + head(): En yüksek oy alan ilk 10 parfüm
df.sort_values('Rating Count', ascending=False)[['Perfume', 'Brand', 'Rating Count', 'Rating Value']].head(10)

## 7. Makine Öğrenmesi – Regresyon Modelleri

**Hedef değişken:** `Rating Value`

**Özellikler:** Üst nota, alt nota, marka


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Eksik değerleri doldur
df['Top']  = df['Top'].fillna('unknown')
df['Base'] = df['Base'].fillna('unknown')

# Üst ve alt notaları TF-IDF ile vektörleştir
tfidf_top  = TfidfVectorizer(max_features=200)
tfidf_base = TfidfVectorizer(max_features=200)

X_top  = tfidf_top.fit_transform(df['Top'])
X_base = tfidf_base.fit_transform(df['Base'])

# Marka için LabelEncoder
le_brand = LabelEncoder()
brand_encoded = le_brand.fit_transform(df['Brand']).reshape(-1, 1)

from scipy.sparse import csr_matrix
X = hstack([X_top, X_base, csr_matrix(brand_encoded)])
y = df['Rating Value'].values

# Eğitim / Test ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Eğitim seti:", X_train.shape, "| Test seti:", X_test.shape)

## 8. Model Eğitimi ve Değerlendirme

In [63]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Modeli eğit, test et ve metrikleri yazdır."""
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    mae   = mean_absolute_error(y_te, preds)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    r2    = r2_score(y_te, preds)
    print(f"--- {name} ---")
    print(f"  MAE : {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²  : {r2:.4f}\n")
    return model, r2

# Üç farklı regresyon modeli dene
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression' : Ridge(alpha=1.0),
    'Lasso Regression' : Lasso(alpha=0.01, max_iter=2000)
}

best_model, best_r2, best_name = None, -999, ''
for name, mdl in models.items():
    fitted, r2 = evaluate_model(name, mdl, X_train, X_test, y_train, y_test)
    if r2 > best_r2:
        best_model, best_r2, best_name = fitted, r2, name

print(f"En iyi model: {best_name} (R² = {best_r2:.4f})")

In [64]:
# Model yorumu
print("""
Model Değerlendirme Yorumu:
- MAE: Tahmin edilen puan ile gerçek puan arasındaki ortalama mutlak fark.
  0.5'in altında olması kabul edilebilir (5'lik skala üzerinde).
- RMSE: Büyük hataları daha fazla cezalandırır; MAE'den yüksek çıkması
  bazı uç tahminlerin olduğunu gösterir.
- R² Skoru: Modelin varyansı ne ölçüde açıkladığını gösterir.
  Marka ve nota bilgisinin puanı tek başına tam belirleyemediğini
  (diğer faktörler: koku kalitesi, tanıtım) bu skor ortaya koyar.
""")

In [65]:
# Modeli ve encoder'ları kaydet (Tkinter uygulaması için)
joblib.dump(best_model,  'model.pkl')
joblib.dump(tfidf_top,   'tfidf_top.pkl')
joblib.dump(tfidf_base,  'tfidf_base.pkl')
joblib.dump(le_brand,    'le_brand.pkl')
print("Model ve encoder'lar kaydedildi.")

## 9. Tkinter Arayüzü Önizlemesi

Arayüz için `final.py` dosyasını çalıştırılacaktır:
```bash
python final.py
```
Kullanıcı **Üst Nota**, **Alt Nota** ve **Marka** girer model 5 üzerinden tahmini puanı döndürür.